# Occupational Employment Statistics Data Ingestion

The purpose of this program is to provide data profiling, cleaning, and ingestion to statistics on Occupational Employment in the United States compiled by the Bureau of Labor Statistics. 

### Data Mapping

##### Retrieve Data

In [ ]:
import org.apache.spark.sql.types._
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val oeAreaPath        = "oe/oe.area"
val oeAreaTypePath    = "oe/oe.areatype"
val oeCurrentDataPath = "oe/oe.data.0.Current"
val oeAllDataPath     = "oe/oe.data.1.AllData"
val oeDataTypePath    = "oe/oe.datatype"
val oeFootnotePath    = "oe/oe.footnote"
val oeIndustryPath    = "oe/oe.industry"
val oeOccupationPath  = "oe/oe.occupation"
val oeReleasePath     = "oe/oe.release"
val oeSeasonalPath    = "oe/oe.seasonal"
val oeSectorPath      = "oe/oe.sector"
val oeSeriesPath      = "oe/oe.series"



##### Area

This dataframe assoociates a state code, areatype code, and area code to a specific location. The data was slightly modified to add the name of the state a location was in to each entry.

In [4]:
val rawAreaDF = spark.read
  .option("header", "true")
  .option("delimiter", "\t")
  .option("inferSchema", true)
  .option("escape", "\"")
  .csv(oeAreaPath)

val stateDF = rawAreaDF.filter($"areatype_code" =!= "M").select("state_code", "area_name").withColumnRenamed("area_name","state_name")

val cleanedAreaDF = rawAreaDF.join(broadcast(stateDF), Seq("state_code"))

z.show(cleanedAreaDF)

##### Area Type

This dataframe associates a area type code to a national, state, or metro/non-metro level

In [5]:
val rawAreaTypeDF = spark.read
    .option("header", "true")
    .option("delimiter", "\t")
    .option("inferSchema", true)
    .option("escape", "\"")
    .csv(oeAreaTypePath)

val cleanedAreaTypeDF = rawAreaTypeDF

z.show(cleanedAreaTypeDF)

##### Data Type

This Dataframe associates a code to specific occupational data type like employment or wage

In [6]:
val rawDataTypeDF = spark.read
    .option("header", "true")
    .option("delimiter", "\t")
    .option("inferSchema", true)
    .option("escape", "\"")
    .csv(oeDataTypePath)

val cleanedDataTypeDF = rawDataTypeDF

z.show(cleanedDataTypeDF)

##### Footnote

This dataframe associates a footnote number to its footnote

In [7]:
val rawFootnoteDF = spark.read
    .option("header", "true")
    .option("delimiter", "\t")
    .option("inferSchema", true)
    .option("escape", "\"")
    .csv(oeFootnotePath)

val cleanedFootnoteDF = rawFootnoteDF

z.show(cleanedFootnoteDF)

##### Industry

This dataframe associates a code to a specific industry

In [8]:
val rawIndustryDF = spark.read
    .option("header", "true")
    .option("delimiter", "\t")
    .option("inferSchema", true)
    .option("escape", "\"")
    .csv(oeIndustryPath)
    
val cleanedIndustryDF = rawIndustryDF
    .select("industry_code", "industry_name")    
    
z.show(cleanedIndustryDF)

##### Occupation

This Dataframe associates a occupation code to a job title and description. The data was modified to add information about the relevant categories a job falls under.

In [9]:
val rawOccupationDF = spark.read
    .option("header", "true")
    .option("delimiter", "\t")
    .option("inferSchema", true)
    .option("escape", "\"")
    .csv(oeOccupationPath)
    
val occupationCategoryDF = rawOccupationDF
    .filter($"display_level" === 0)
    .select(floor(col("occupation_code").divide(10000)) as "category_code", col("occupation_name") as "occupation_category_name")
    
val occupationSubCategoryDF = rawOccupationDF
    .filter($"display_level" < 2)
    .select(floor(col("occupation_code").divide(1000)) as "subcategory_code", col("occupation_name") as "occupation_subcategory_name")
    
val cleanedOccupationDF = rawOccupationDF
    .select("occupation_code", "occupation_name", "occupation_description")
    .withColumn("category_code", floor(col("occupation_code").divide(10000)))
    .withColumn("subcategory_code", floor(col("occupation_code").divide(1000)))
    .join(broadcast(occupationCategoryDF), Seq("category_code"))
    .drop("category_code")
    .join(broadcast(occupationSubCategoryDF), Seq("subcategory_code"))
    .drop("subcategory_code")
z.show(cleanedOccupationDF)

##### Release

This Dataframe states the release time of this dataset, and was deemed irrelevant to the project

In [10]:
val rawReleaseDF = spark.read
    .option("header", "true")
    .option("delimiter", "\t")
    .option("inferSchema", true)
    .option("escape", "\"")
    .csv(oeReleasePath)
    
z.show(rawReleaseDF)

##### Seasonal

This Dataframe states if the data was seasonally adjusted, and was deemed irrelevant to the project.

In [11]:
val rawSeasonalDF = spark.read
    .option("header", "true")
    .option("delimiter", "\t")
    .option("inferSchema", true)
    .option("escape", "\"")
    .csv(oeSeasonalPath)
    
z.show(rawSeasonalDF)

##### Sector

This Dataframe holds information on the work sector associated with a specific code.

In [12]:
val rawSectorDF = spark.read
    .option("header", "true")
    .option("delimiter", "\t")
    .option("inferSchema", true)
    .option("escape", "\"")
    .csv(oeSectorPath)

val cleanedSectorDF = rawSectorDF
   
z.show(cleanedSectorDF)

### Data Files

##### Series Data

In [13]:
val rawSeriesDF = spark.read
    .option("header", "true")
    .option("delimiter", "\t")
    .option("inferSchema", true)
    .option("escape", "\"")
    .csv(oeSeriesPath)
    
z.show(rawSeriesDF)

In [14]:
rawSeriesDF.columns.length

In [15]:
rawSeriesDF.printSchema

In [16]:
val baseSeriesDF = rawSeriesDF.select(
    "series_id                     ",
    "areatype_code", 
    "industry_code", 
    "occupation_code", 
    "datatype_code", 
    "state_code", 
    "area_code", 
    "sector_code"
    )
    .withColumnRenamed("series_id                     ", "series_id")

baseSeriesDF.cache().count

In [17]:
z.show(baseSeriesDF)

In [18]:
baseSeriesDF.printSchema

In [19]:
z.show(baseSeriesDF.describe())

In [20]:
z.show(baseSeriesDF.summary())

Mapping files were applied to the Series Dataframe through a sequence of broadcast joins

In [21]:
val completeSeriesDF = baseSeriesDF
    .join(
        broadcast(cleanedAreaTypeDF), 
        Seq("areatype_code")
    )
    .drop("areatype_code")
    .join(
        broadcast(cleanedIndustryDF), 
        Seq("industry_code")
    )
    .drop("industry_code")
    .join(
        broadcast(cleanedOccupationDF), 
        Seq("occupation_code")
    )
    .drop("occupation_code")
    .join(
        broadcast(cleanedDataTypeDF), 
        Seq("datatype_code")
    )
    .drop("datatype_code")
    .join(
        broadcast(cleanedAreaDF), 
        Seq("state_code", "area_code")
    )
    .drop("state_code", "area_code", "areatype_code")
    .join(
        broadcast(cleanedSectorDF), 
        Seq("sector_code")
    )
    .drop("sector_code")
    

z.show(completeSeriesDF)

In [22]:
z.show(rawSeriesDF.summary())

### All Data

In [23]:
val rawAllDataDF = spark.read
    .option("header", "true")
    .option("delimiter", "\t")
    .option("inferSchema", true)
    .option("escape", "\"")
    .csv(oeAllDataPath)
    
z.show(rawAllDataDF)

In [24]:
z.show(rawAllDataDF.summary())

In [25]:
rawAllDataDF.printSchema

Titles are fixed due to inferred schema reading errors

In [26]:
val baseDataDF = rawAllDataDF.select(
    "series_id                     ",
    "       value", 
    "footnote_codes"
    )
    .withColumnRenamed("series_id                     ", "series_id")
    .withColumnRenamed("       value","value")
    .withColumnRenamed("footnote_codes", "footnote_code")

baseSeriesDF.cache().count

In [27]:
z.show(baseDataDF)

In [28]:
baseDataDF.printSchema

Change the value from inferred type String to its actual type Double, and get the datatype from the end of the series id.

In [29]:
val fixedDataDF = baseDataDF
    .withColumn("value", $"value".cast("double"))
    .withColumn("datatype", substring(col("series_id"),24,2))

z.show(fixedDataDF)

Impute values based on footnote 5 stating `This wage is equal to or greater than $115.00 per hour or $239,200 per year.`

For hourly wages these entries are imputed as `$120.00` while for annual wages these entries are imputed as `$240,000`. The purpose is to connote the value of a wage that is above those thresholds to a degree while still connoting these entries as lacking information.

Other null values are simply removed rather than being imputed.

In [30]:
val imputedDataDF = fixedDataDF
    .withColumn(
        "value",
        when($"footnote_code"===5 && ($"datatype"==="03" || $"datatype"==="06" || $"datatype"==="07" || $"datatype"==="08" || $"datatype"==="09" || $"datatype"==="10"), 
            120.0
            )
        .when($"footnote_code"===5 && ($"datatype"==="04" || $"datatype"==="11" || $"datatype"==="12" || $"datatype"==="13" || $"datatype"==="14" || $"datatype"==="15"), 
            240000.0
            )
        .otherwise($"value")
    )
    .filter($"value".isNotNull)
    .drop("footnote_code")
    .drop("datatype")

z.show(imputedDataDF)

### Completing the Data

##### Joining the Data

The partitions of each file are examined, and then set to 10 for the ease of joining.

In [31]:
imputedDataDF.rdd.getNumPartitions


In [32]:
completeSeriesDF.rdd.getNumPartitions

In [33]:
val bucketDataDF = imputedDataDF.repartition(10, $"series_id").cache()
val bucketSeriesDF = completeSeriesDF.repartition(10, $"series_id").cache()

In [34]:
val completeOEDF = bucketSeriesDF.join(bucketDataDF, Seq("series_id"))

z.show(completeOEDF)

In [35]:
completeOEDF.cache().count

##### Saving the Data

In [36]:
val finalOEDF = completeOEDF.drop("series_id")

z.show(finalOEDF)

In [37]:
finalOEDF.write.mode("overwrite").option("compression", "snappy").save("cleanedOEData.parquet")